# Chongchen Jiang — Model Development and Validation
## 1. Validation Strategy: Initial Data Inspection

**Objective:** Examine the transaction structure and repeated
property sales before choosing a validation strategy.

In [1]:
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error


def evaluate_regression(y_true, y_pred):
    return {
        "MSE": mean_squared_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred)
    }

def format_metric_table(results):
    formatted = results.copy()
    formatted["MSE"] = formatted["MSE"].map(lambda value: f"{value:,.2f}")
    formatted["R²"] = formatted["R²"].map(lambda value: f"{value:.4f}")
    formatted["MAE"] = formatted["MAE"].map(lambda value: f"{value:,.2f}")
    return formatted

# Load raw transaction data
sales = pd.read_csv("data/sales.csv")

# Inspect the dataset
print("Dataset shape:", sales.shape)
print("\nColumns:")
print(sales.columns.tolist())

print("\nData types:")
print(sales.dtypes)

# Check date coverage
sales["sale_date"] = pd.to_datetime(
    sales["sale_date"], errors="coerce"
)

print("\nTransactions by year:")
print(sales["sale_date"].dt.year.value_counts().sort_index())

# Check duplicate transactions
print("\nDuplicate transaction IDs:")
print(sales["transaction_id"].duplicated().sum())

# Check repeated property sales
property_counts = sales.groupby("property_id").size()

print("\nProperties with multiple sales:")
print((property_counts > 1).sum())

Dataset shape: (109307, 4)

Columns:
['property_id', 'transaction_id', 'sale_date', 'sale_price']

Data types:
property_id         int64
transaction_id      int64
sale_date          object
sale_price        float64
dtype: object

Transactions by year:
2023    37520
2024    35465
2025    36322
Name: sale_date, dtype: int64

Duplicate transaction IDs:
0

Properties with multiple sales:
9774


In [2]:

# Check property overlap between 2023 and 2024

train_2023 = sales[
    sales["sale_date"].dt.year == 2023
].copy()

val_2024 = sales[
    sales["sale_date"].dt.year == 2024
].copy()

# Properties observed in the training period
train_property_ids = set(train_2023["property_id"])

# Identify whether each validation transaction
# belongs to a previously observed property
val_2024["seen_property"] = (
    val_2024["property_id"].isin(train_property_ids)
)

n_seen = val_2024["seen_property"].sum()
n_total = len(val_2024)
n_unseen = n_total - n_seen

print("2024 validation transactions:", n_total)
print("Seen-property transactions:", n_seen)
print("Unseen-property transactions:", n_unseen)
print("Seen-property proportion:", round(n_seen / n_total * 100, 2), "%")
print("Distinct seen property IDs:", val_2024.loc[val_2024["seen_property"], "property_id"].nunique())
print("Training and validation transaction counts:", len(train_2023), n_total)
print("2025 transactions reserved for final test:", (sales["sale_date"].dt.year == 2025).sum())


2024 validation transactions: 35465
Seen-property transactions: 2498
Unseen-property transactions: 32967
Seen-property proportion: 7.04 %
Distinct seen property IDs: 2458
Training and validation transaction counts: 37520 35465
2025 transactions reserved for final test: 36322


In [3]:

# Compare sale-price distributions between
# seen and unseen properties in 2024

price_comparison = (
    val_2024.groupby("seen_property")["sale_price"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        p25=lambda x: x.quantile(0.25),
        p75=lambda x: x.quantile(0.75),
        p90=lambda x: x.quantile(0.90),
        p95=lambda x: x.quantile(0.95),
        p99=lambda x: x.quantile(0.99),
        max="max"
    )
)

price_comparison.index = price_comparison.index.map({
    False: "Unseen properties",
    True: "Seen properties"
})

display(price_comparison.round(2))


,count,mean,median,p25,p75,p90,p95,p99,max
seen_property,,,,,,,,,
Unseen properties,32967,447977.75,345500.0,230000.0,515550.0,835000.0,1190000.0,2149934.00,15250000.0
Seen properties,2498,440161.06,331500.0,250000.0,460000.0,720000.0,1131675.0,2352325.92,4575000.0


In [4]:

# Global Mean Baseline
# Train using 2023 prices only

global_mean = train_2023["sale_price"].mean()

# Predict the same value for all 2024 transactions
val_2024["baseline_pred"] = global_mean

# Overall performance
overall_metrics = evaluate_regression(
    val_2024["sale_price"], val_2024["baseline_pred"]
)
overall_results = pd.DataFrame([{
    "Baseline": "Global Mean",
    **overall_metrics
}]).set_index("Baseline").rename(columns={"R2": "R²"})

# Seen vs unseen metrics are diagnostic; counts refer to transactions
subgroup_rows = []
for seen, group in val_2024.groupby("seen_property"):
    subgroup_rows.append({
        "Group": "Seen properties" if seen else "Unseen properties",
        "n_transactions": len(group),
        "share_pct": len(group) / len(val_2024) * 100,
        **evaluate_regression(group["sale_price"], group["baseline_pred"])
    })
subgroup_metrics = pd.DataFrame(subgroup_rows).set_index("Group")
subgroup_metrics = subgroup_metrics.rename(columns={"R2": "R²"})

print("Training global mean:", round(global_mean, 2))
print("\n2024 Global Mean validation metrics:")
display(format_metric_table(overall_results))
print("\nMetrics by seen/unseen group:")
display(format_metric_table(subgroup_metrics))


Training global mean: 406135.87

2024 Global Mean validation metrics:


,MSE,R²,MAE
Baseline,,,
Global Mean,"180,697,179,623.22",-0.0095,"237,022.94"



Metrics by seen/unseen group:


,n_transactions,share_pct,MSE,R²,MAE
Group,,,,,
Unseen properties,32967,92.956436,"181,283,422,153.43",-0.0098,"238,565.02"
Seen properties,2498,7.043564,"172,960,327,143.81",-0.0067,"216,671.61"



### 3c. Validation Design and Global Mean Baseline Findings

| Evaluation | Question answered |
|---|---|
| Overall Validation | How does the model perform on all 2024 transactions? |
| Unseen Properties | How does it perform when the property did not appear in 2023 training? |
| Seen Properties | Does performance differ for properties present in 2023 training? |
| Property-disjoint Sensitivity Check | Does the main result change when evaluation is restricted to properties absent from training? |


The Global Mean Baseline predicts the 2023 training mean ($406,135.87)
for every property, regardless of property identity.

All models are compared on the same validation set using MSE, R² and MAE, with preprocessing fitted within each training split and applied to the corresponding validation data.
MSE is lower-is-better in squared-dollar units; R² is higher-is-better and can be negative; MAE is lower-is-better in dollars.

On 2024, Global Mean MSE is $180,697,179,623.22, R² is -0.0095, and MAE is $237,022.94.
Seen transactions have MSE $172,960,327,143.81, R² -0.0067, and MAE $216,671.61 (n=2,498);
unseen transactions have MSE $181,283,422,153.43, R² -0.0098, and MAE $238,565.02 (n=32,967).
Median prices are similar ($331,500 seen; $345,500 unseen). Unseen sales have
a higher 90th and 95th percentile, while the seen group has a higher 99th
percentile; the unseen maximum is much higher ($15.25m vs $4.575m). The
maximum is sensitive to rare extreme sales, so these summaries show differing
tails rather than a uniformly higher price distribution for either group.

For this Global Mean baseline, the property-disjoint sensitivity result is
the Unseen Properties row above: evaluate the same model and predictions
after excluding validation transactions whose property IDs occurred in
training. The unseen-only MSE is $586,242,530.21 (0.32%) higher than overall MSE;
unseen-only MAE is $1,542.08 (0.65%) higher than overall MAE. This is a
sensitivity view of the same time split, not a new split or a refit.

The baseline predicts the same value for every transaction and does not use
property IDs as predictive features.

Because this baseline uses no property-level inputs, the error difference
does not establish that the repeated-property overlap caused leakage. The
different group price distributions are relevant context for interpretation.

Therefore, lower errors for previously seen properties
would not, by themselves, establish data leakage.
The 7.04% overlap alone does not establish that validation is optimistic;
these baseline results show only a small difference in aggregate errors.
Unseen does not mean without recent sale: this data identifies overlap with
2023 training transactions only and cannot establish a complete sale history.
This is a diagnostic for the time-based validation set; 2025 remains
reserved for final testing.


In [5]:

# Inspect property attributes before joining

properties = pd.read_csv("data/properties_2022.csv")

print("Property dataset shape:", properties.shape)

print("\nPossible location columns:")
print([
    col for col in properties.columns
    if "town" in col.lower()
    or "location" in col.lower()
    or "latitude" in col.lower()
    or "longitude" in col.lower()
])

print("\nDuplicate property IDs:")
print(properties["property_id"].duplicated().sum())


Property dataset shape: (98899, 98)

Possible location columns:
['loc_longitude', 'loc_latitude', 'meta_township_code']

Duplicate property IDs:
0


C:\Users\众神已死，我是太阳\AppData\Local\Temp\ipykernel_36540\3200835891.py:3: DtypeWarning: Columns (37) have mixed types. Specify dtype option on import or set low_memory=False.
  properties = pd.read_csv("data/properties_2022.csv")


In [6]:

# Check the sales-to-property join before building
# the Township Median Baseline

property_location = properties[
    ["property_id", "meta_township_code"]
].copy()

sales_with_location = sales.merge(
    property_location,
    on="property_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Original sales rows:", len(sales))
print("Rows after joining:", len(sales_with_location))

print("\nJoin status:")
print(sales_with_location["_merge"].value_counts())

print("\nMissing township codes:")
print(sales_with_location["meta_township_code"].isna().sum())

print("\nNumber of distinct townships:")
print(sales_with_location["meta_township_code"].nunique())

print("\nSample township codes:")
print(sales_with_location["meta_township_code"].dropna().unique()[:10])


Original sales rows: 109307
Rows after joining: 109307

Join status:
both          109307
left_only          0
right_only         0
Name: _merge, dtype: int64

Missing township codes:
0

Number of distinct townships:
38

Sample township codes:
[25 21 30 70 28 37 12 29 16 77]


In [7]:

import numpy as np
import pandas as pd

# --------------------------------------------------
# Research questions:
# 1. Does Mean or Median work better as a baseline?
# 2. Does adding Township information improve prediction?
#
# Preliminary split:
# Train: 2023
# Validation: 2024
# Final test: 2025 (not used here)
# --------------------------------------------------

df = sales_with_location.copy()
df["sale_date"] = pd.to_datetime(df["sale_date"])

train = df[df["sale_date"].dt.year == 2023].copy()
val = df[df["sale_date"].dt.year == 2024].copy()

# 1. Learn all baseline statistics from TRAIN only

global_mean = train["sale_price"].mean()
global_median = train["sale_price"].median()

township_stats = (
    train.groupby("meta_township_code")["sale_price"]
    .agg(["mean", "median"])
)

# 2. Predict on the SAME validation observations

val["pred_global_mean"] = global_mean
val["pred_global_median"] = global_median

val["pred_township_mean"] = (
    val["meta_township_code"]
    .map(township_stats["mean"])
    .fillna(global_mean)
)

val["pred_township_median"] = (
    val["meta_township_code"]
    .map(township_stats["median"])
    .fillna(global_median)
)

# 3. Evaluate with identical metrics

baseline_columns = {
    "Global Mean": "pred_global_mean",
    "Global Median": "pred_global_median",
    "Township Mean": "pred_township_mean",
    "Township Median": "pred_township_median"
}

results = []

for name, prediction_col in baseline_columns.items():
    metrics = evaluate_regression(
        val["sale_price"],
        val[prediction_col]
    )

    results.append({
        "Baseline": name,
        **metrics
    })

baseline_results = pd.DataFrame(results).set_index("Baseline")
baseline_results = baseline_results.rename(columns={"R2": "R²"})

print("2023 training mean:", round(global_mean, 2))
print("2023 training median:", round(global_median, 2))

print("\nTownships in training:", len(township_stats))
print(
    "Validation observations in unseen townships:",
    (~val["meta_township_code"].isin(township_stats.index)).sum()
)

print("\n2024 Validation Results:")
display(format_metric_table(baseline_results))


2023 training mean: 406135.87
2023 training median: 315000.0

Townships in training: 38
Validation observations in unseen townships: 0

2024 Validation Results:


,MSE,R²,MAE
Baseline,,,
Global Mean,"180,697,179,623.22",-0.0095,"237,022.94"
Global Median,"196,529,164,386.04",-0.0980,"231,085.44"
Township Mean,"102,140,055,434.59",0.4294,"170,855.63"
Township Median,"113,899,897,274.75",0.3637,"170,953.95"


### 3a. Baseline Comparison — 2024 Validation

**Table 1. Baseline performance on the 2024 validation set.**

| Baseline | MSE (USD²) | R² | MAE (USD) |
|---|---:|---:|---:|
| Global Mean | 180,697,179,623.22 | -0.0095 | 237,022.94 |
| Global Median | 196,529,164,386.04 | -0.0980 | 231,085.44 |
| Township Mean | 102,140,055,434.59 | 0.4294 | 170,855.63 |
| Township Median | 113,899,897,274.75 | 0.3637 | 170,953.95 |

The Global Mean baseline predicts the 2023 training mean sale price ($406,135.87) for every 2024 transaction, providing a simple benchmark. MSE is lower-is-better and measured in squared dollars; R² is higher-is-better and can be negative; MAE is lower-is-better and measured in dollars.

Both township baselines outperform their corresponding global baselines on all three metrics in this validation comparison. Township Mean has the lowest MSE and MAE and the highest R². Its MAE is only $98.32 lower than Township Median, so the practical importance and stability of that small MAE difference remain uncertain. These are baseline results on the 2024 validation set; the 2025 test set remains held out.